# 02 - RAGAS comparison: run1 vs run2

An **offline** demo that reads the saved score files (`results/run*_scores.csv`) and the saved retrieval
inputs. The scores were produced by `python -m src.evaluate` with an Azure OpenAI judge (gpt-4.1-mini,
temperature 0). Nothing here calls a model.

**Reading the numbers honestly**
- Averages use the 8 in-scope questions. The 2 out-of-scope questions are excluded: RAGAS scores a correct
  refusal as 0 for Response Relevancy.
- A difference smaller than 0.05 is marked *within noise* (a convention, not a statistical test; with 8
  questions, one question moving by 0.4 shifts an average by 0.05).
- The judge makes systematic mistakes (see the last cell), so read examples before trusting a score.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter was started in the repo root or in notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.metrics import summarize

testset = json.loads((ROOT / "data/testsets/testset_v1.json").read_text(encoding="utf-8"))
inputs = {
    "run1": json.loads((ROOT / "results/run1_baseline_inputs.json").read_text(encoding="utf-8")),
    "run2": json.loads((ROOT / "results/run2_semantic_inputs.json").read_text(encoding="utf-8")),
}
s1 = pd.read_csv(ROOT / "results/run1_baseline_scores.csv")
s2 = pd.read_csv(ROOT / "results/run2_semantic_scores.csv")
print(len(s1), "and", len(s2), "scored questions")

## In-scope averages: 3 exact metrics + 4 RAGAS metrics

In [ ]:
rows = []
_, h1, m1, f1 = summarize(inputs["run1"], testset)
_, h2, m2, f2 = summarize(inputs["run2"], testset)
rows += [("hit rate@3", h1, h2), ("MRR", m1, m2), ("full coverage", f1, f2)]

ragas_metrics = {
    "faithfulness": "faithfulness",
    "answer_relevancy": "response relevancy",
    "llm_context_precision_with_reference": "context precision",
    "context_recall": "context recall",
}
for column, label in ragas_metrics.items():
    a = s1.loc[s1["type"] != "out_of_scope", column].mean()
    b = s2.loc[s2["type"] != "out_of_scope", column].mean()
    rows.append((label, a, b))

table = pd.DataFrame(rows, columns=["metric", "run1", "run2"])
table["diff"] = table["run2"] - table["run1"]
def flag(d):
    if abs(d) < 1e-9:
        return "unchanged"
    if abs(d) < 0.05:
        return "within noise"
    return "better" if d > 0 else "worse"


table["note"] = table["diff"].apply(flag)
table.round(3)

## Per question (RAGAS)

In [ ]:
columns = list(ragas_metrics)
per_question = s1[["id", "type"] + columns].merge(s2[["id"] + columns], on="id", suffixes=(" run1", " run2"))
per_question.round(2)

## Where the judge looks unreliable
- **q06:** chunks 4 and 5 contain exactly the reference facts and the answer is correct and cited, yet
  context precision is 0.00 in both runs. Identical inputs gave identical wrong verdicts at temperature 0,
  so the error is systematic: averaging repeated runs would not remove it.
- **q07:** context precision is 0.33 in both runs although chunk 8 plainly holds the fracture-risk facts.
- **q02:** a correct, cited answer gets response relevancy 0.00 (likely flagged "noncommittal").
- **q08:** the same refusal scored faithfulness 1.0 in run1 and 0.0 in run2.

The same model (gpt-4.1-mini) generates the answers and judges them, so self-preference bias is possible.
A stronger, different judge would be a separate experiment.